# Tech Challenge - Classifica??o de C?ncer de Mama

## Contexto
Este notebook apresenta uma proposta inicial de aplica??o de Machine Learning para apoiar a triagem de risco em sa?de da mulher, utilizando o dataset `breast-cancer.csv`. A organiza??o do trabalho segue a estrutura solicitada no Tech Challenge, contemplando explora??o dos dados, pr?-processamento, modelagem, avalia??o e explicabilidade.

## Objetivo
O objetivo ? comparar t?cnicas de classifica??o capazes de prever o diagn?stico de c?ncer de mama (`M` = maligno, `B` = benigno), com aten??o especial ?s m?tricas mais adequadas ao contexto m?dico. Como se trata de um problema de triagem, reduzir falsos negativos ? especialmente importante, j? que classificar um caso maligno como benigno pode atrasar a investiga??o cl?nica.

## Estrat?gia adotada
1. Carregar e inspecionar o dataset.
2. Realizar an?lise explorat?ria com estat?sticas e visualiza??es.
3. Executar o pr?-processamento com justificativas.
4. Treinar e comparar tr?s modelos de classifica??o.
5. Avaliar o desempenho com valida??o cruzada e conjunto de teste.
6. Interpretar o modelo de melhor resultado com feature importance e SHAP.

## Observa??o metodol?gica
A constru??o do notebook foi guiada por pr?ticas usuais de an?lise de dados e Machine Learning com `pandas`, `matplotlib`, `seaborn` e `scikit-learn`, mantendo um fluxo compat?vel com o conte?do trabalhado ao longo da disciplina.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid', palette='Blues')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

try:
    import shap
    SHAP_DISPONIVEL = True
except ImportError:
    SHAP_DISPONIVEL = False
    print("Biblioteca 'shap' nao encontrada. As celulas de SHAP ficarao opcionais.")

## 1. Carregamento dos dados
O primeiro passo consiste em carregar o dataset e observar sua estrutura geral. Essa etapa ? importante para confirmar o volume de registros, as colunas dispon?veis e a natureza do problema que ser? tratado.

In [ ]:
df = pd.read_csv('dataset/breast-cancer.csv')

print(f'Dimensoes do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas')
display(df.head())

In [ ]:
df.info()

### Leitura inicial
- O dataset possui 569 registros.
- A vari?vel alvo ? `diagnosis`, com duas classes: `M` (maligno) e `B` (benigno).
- A coluna `id` funciona apenas como identificador do exame e n?o representa uma caracter?stica cl?nica ?til para o treinamento.
- As demais vari?veis s?o num?ricas e descrevem medidas extra?das dos exames.

## 2. An?lise explorat?ria dos dados
Nesta etapa, s?o examinadas estat?sticas descritivas, distribui??es e rela??es entre vari?veis. O objetivo ? compreender melhor o comportamento dos dados antes do processo de modelagem.

In [ ]:
display(df.describe().T)

In [ ]:
class_dist = df['diagnosis'].value_counts().rename_axis('diagnosis').reset_index(name='count')
class_dist['percentual'] = 100 * class_dist['count'] / class_dist['count'].sum()
display(class_dist)

fig, ax = plt.subplots()
sns.countplot(data=df, x='diagnosis', order=['B', 'M'], ax=ax)
ax.set_title('Distribuicao das classes do diagnostico')
ax.set_xlabel('Diagnostico')
ax.set_ylabel('Quantidade')
plt.show()

### Coment?rio
H? mais casos benignos do que malignos. Ainda assim, o desbalanceamento n?o ? severo. Isso refor?a a import?ncia de utilizar separa??o estratificada entre treino e teste e de acompanhar, al?m da acur?cia, m?tricas como `recall` e `f1-score`.

In [ ]:
selected_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feature in zip(axes.flatten(), selected_features):
    sns.boxplot(data=df, x='diagnosis', y=feature, order=['B', 'M'], ax=ax)
    ax.set_title(f'{feature} por diagnostico')
plt.tight_layout()
plt.show()

### Leitura dos boxplots
Os boxplots ajudam a verificar se algumas medidas apresentam separa??o visual entre casos benignos e malignos. Quando essa diferen?a aparece de forma consistente, ela j? sugere que certas vari?veis morfol?gicas podem contribuir de maneira relevante para a classifica??o.

In [ ]:
corr = df.drop(columns=['id']).copy()
corr['diagnosis'] = corr['diagnosis'].map({'B': 0, 'M': 1})
corr_matrix = corr.corr(numeric_only=True)

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title('Matriz de correlacao')
plt.show()

In [ ]:
target_corr = corr_matrix['diagnosis'].drop('diagnosis').sort_values(key=np.abs, ascending=False)
display(target_corr.head(10).to_frame('correlacao_com_diagnosis'))

### Coment?rio sobre correla??o
A matriz de correla??o permite identificar tanto vari?veis fortemente relacionadas entre si quanto aquelas mais associadas ao diagn?stico. Essa leitura ? ?til para a interpreta??o dos resultados, para a identifica??o de poss?veis redund?ncias e para justificar o uso de modelos mais sens?veis ? escala e ? proximidade, como o KNN.

## 3. Pr?-processamento dos dados
O pr?-processamento foi organizado em etapas objetivas, sempre com justificativas compat?veis com o problema analisado.

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
duplicated_rows = df.duplicated().sum()

print('Valores ausentes por coluna:')
display(missing_values.to_frame('missing'))
print(f'Linhas duplicadas: {duplicated_rows}')

### Decis?es de pr?-processamento
1. **Valores ausentes**: o dataset n?o apresenta valores ausentes, portanto n?o foi necess?rio imputar dados na base original. Ainda assim, os pipelines incluem `SimpleImputer` como boa pr?tica de reprodutibilidade.
2. **Inconsist?ncias**: n?o foram observadas inconsist?ncias evidentes de tipo ou formato nas vari?veis num?ricas.
3. **Remo??o da coluna `id`**: essa coluna atua apenas como identificador e n?o traz informa??o cl?nica relevante para a generaliza??o do modelo.
4. **Codifica??o da vari?vel alvo**: a classe `diagnosis` foi convertida para formato num?rico (`B = 0`, `M = 1`) para utiliza??o nos algoritmos.
5. **Escalonamento**: foi aplicado `StandardScaler` nos modelos sens?veis ? escala, como Regress?o Log?stica e KNN.
6. **Separa??o treino/teste**: foi utilizada divis?o estratificada para preservar a propor??o entre benigno e maligno nos subconjuntos.

In [ ]:
df_model = df.copy()
df_model = df_model.drop(columns=['id'])

label_encoder = LabelEncoder()
df_model['diagnosis_encoded'] = label_encoder.fit_transform(df_model['diagnosis'])

X = df_model.drop(columns=['diagnosis', 'diagnosis_encoded'])
y = df_model['diagnosis_encoded']

print('Visualizacao dos dados apos remocao do ID e codificacao do alvo:')
display(df_model.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print('Dimensoes dos conjuntos:')
print(f'Treino: {X_train.shape}')
print(f'Teste: {X_test.shape}')

In [ ]:
numeric_features = X.columns.tolist()

scaled_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features)
    ],
    remainder='drop'
)

preview_scaler = Pipeline([
    ('preprocessor', scaled_preprocessor)
])

X_train_scaled_preview = preview_scaler.fit_transform(X_train)
X_train_scaled_preview = pd.DataFrame(X_train_scaled_preview, columns=numeric_features, index=X_train.index)

print('Amostra dos dados numericos apos escalonamento (treino):')
display(X_train_scaled_preview.head())

## 4. Modelagem
Foram escolhidos tr?s algoritmos de classifica??o com comportamentos distintos, de modo a permitir uma compara??o mais consistente:

- **Regress?o Log?stica**: modelo linear, de f?cil interpreta??o e bastante ?til como refer?ncia inicial em problemas de classifica??o bin?ria.
- **KNN**: modelo baseado em proximidade, sens?vel ? escala das vari?veis e interessante para compara??o com uma abordagem n?o param?trica.
- **Random Forest**: conjunto de ?rvores de decis?o capaz de capturar rela??es n?o lineares e fornecer medidas de import?ncia das vari?veis.

In [ ]:
models = {
    'Regressao Logistica': Pipeline([
        ('preprocessor', scaled_preprocessor),
        ('model', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', scaled_preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', ColumnTransformer(
            transformers=[
                ('num', Pipeline([
                    ('imputer', SimpleImputer(strategy='median'))
                ]), numeric_features)
            ],
            remainder='drop'
        )),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42
        ))
    ])
}

models

## 5. Valida??o cruzada
Antes da avalia??o final no conjunto de teste, cada modelo ? analisado por valida??o cruzada estratificada. Essa etapa reduz a depend?ncia de uma ?nica divis?o da base e ajuda a estimar a estabilidade do desempenho.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1']

cv_results = []

for model_name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )
    cv_results.append({
        'modelo': model_name,
        'accuracy_medio': scores['test_accuracy'].mean(),
        'precision_medio': scores['test_precision'].mean(),
        'recall_medio': scores['test_recall'].mean(),
        'f1_medio': scores['test_f1'].mean(),
        'accuracy_std': scores['test_accuracy'].std(),
        'recall_std': scores['test_recall'].std(),
        'f1_std': scores['test_f1'].std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    by=['recall_medio', 'f1_medio', 'accuracy_medio'],
    ascending=False
).reset_index(drop=True)

display(cv_results_df)

### Crit?rio de compara??o
Como o problema envolve apoio ? triagem de um poss?vel c?ncer, o `recall` da classe maligna recebe aten??o especial. Em seguida, o `f1-score` ajuda a equilibrar `precision` e `recall`. A `accuracy` continua sendo uma m?trica relevante, mas n?o deve ser usada isoladamente em um contexto m?dico.

## 6. Treinamento final e avaliação no conjunto de teste

In [ ]:
test_results = []
fitted_models = {}

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline

    y_pred = pipeline.predict(X_test)

    test_results.append({
        'modelo': model_name,
        'accuracy_teste': accuracy_score(y_test, y_pred),
        'precision_teste': precision_score(y_test, y_pred),
        'recall_teste': recall_score(y_test, y_pred),
        'f1_teste': f1_score(y_test, y_pred)
    })

test_results_df = pd.DataFrame(test_results).sort_values(
    by=['recall_teste', 'f1_teste', 'accuracy_teste'],
    ascending=False
).reset_index(drop=True)

display(test_results_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, pipeline) in zip(axes, fitted_models.items()):
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benigno', 'Maligno'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(model_name)

plt.tight_layout()
plt.show()

In [ ]:
for model_name, pipeline in fitted_models.items():
    y_pred = pipeline.predict(X_test)
    print('=' * 80)
    print(model_name)
    print(classification_report(y_test, y_pred, target_names=['Benigno', 'Maligno']))

### Interpreta??o da avalia??o
Na compara??o final, o principal ponto de aten??o ? a capacidade de identificar corretamente os casos malignos. Quando dois modelos apresentam `accuracy` semelhante, tende a ser mais adequado priorizar aquele com melhor `recall` e `f1-score` para a classe de maior risco.

## 7. Escolha do melhor modelo
A escolha final ser? feita com base no crit?rio adotado para o problema: maior `recall`, seguido por maior `f1-score` e, por fim, maior `accuracy`.

In [ ]:
best_model_name = test_results_df.iloc[0]['modelo']
best_model = fitted_models[best_model_name]

print(f'Melhor modelo segundo o criterio definido: {best_model_name}')
display(test_results_df.head(1))

## 8. Explicabilidade
A etapa de explicabilidade ? importante para tornar os resultados mais transparentes. Neste notebook, foram consideradas duas abordagens complementares:

- **Feature importance**: indica quais vari?veis tiveram maior peso na decis?o do modelo baseado em ?rvores.
- **SHAP**: ajuda a entender como cada atributo contribui para aumentar ou reduzir a previs?o de malignidade.

In [ ]:
if best_model_name == 'Random Forest':
    rf_estimator = best_model.named_steps['model']
    importances = pd.Series(rf_estimator.feature_importances_, index=numeric_features)
    importances = importances.sort_values(ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='Blues_r')
    plt.title('Top 15 variaveis mais importantes - Random Forest')
    plt.xlabel('Importancia')
    plt.ylabel('Variavel')
    plt.show()
else:
    print('O melhor modelo nao foi o Random Forest. Ainda assim, a celula de SHAP abaixo pode ser aplicada a modelos baseados em arvore, se desejado.')

In [ ]:
if SHAP_DISPONIVEL and best_model_name == 'Random Forest':
    X_test_prepared = pd.DataFrame(
        best_model.named_steps['preprocessor'].transform(X_test),
        columns=numeric_features,
        index=X_test.index
    )

    explainer = shap.TreeExplainer(best_model.named_steps['model'])
    shap_values = explainer.shap_values(X_test_prepared)

    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values[..., 1] if getattr(shap_values, 'ndim', 2) == 3 else shap_values

    shap.summary_plot(shap_values_to_plot, X_test_prepared, plot_type='bar', show=False)
    plt.title('SHAP summary plot (bar) - impacto medio das variaveis')
    plt.tight_layout()
    plt.show()

    shap.summary_plot(shap_values_to_plot, X_test_prepared, show=False)
    plt.title('SHAP summary plot - direcao e intensidade do impacto')
    plt.tight_layout()
    plt.show()
else:
    print("SHAP nao executado. Verifique se a biblioteca 'shap' esta instalada e se o melhor modelo foi o Random Forest.")

### Como interpretar a explicabilidade
- No gr?fico de **feature importance**, as vari?veis posicionadas no topo tendem a exercer maior influ?ncia sobre a decis?o do modelo.
- No **SHAP summary plot**, cada ponto representa uma observa??o. A posi??o horizontal indica o impacto da vari?vel na previs?o, enquanto a cor mostra se o valor observado ? mais alto ou mais baixo.
- Em um contexto aplicado, essa leitura contribui para justificar tecnicamente por que o modelo indicou maior ou menor risco.

## 9. Conclus?o cr?tica
Este projeto apresenta um fluxo completo de classifica??o com Machine Learning aplicado a dados de sa?de da mulher. A compara??o entre algoritmos permite selecionar um modelo mais consistente, considerando n?o apenas `accuracy`, mas tamb?m `recall` e `f1-score`, m?tricas particularmente relevantes em cen?rios de triagem de risco.

Do ponto de vista pr?tico, o modelo **n?o deve substituir o m?dico**. Seu uso mais apropriado ? como ferramenta de apoio ? triagem, sinalizando casos que merecem maior aten??o ou prioridade de an?lise. Em um ambiente real, essa solu??o ainda exigiria valida??o adicional, controle de vi?s, monitoramento cont?nuo e integra??o com protocolos cl?nicos.

## Poss?veis evolu??es
1. Ajuste fino de hiperpar?metros com `GridSearchCV`.
2. Teste com outros modelos, como ?rvore de Decis?o, SVM ou XGBoost.
3. An?lise mais aprofundada de sensibilidade e especificidade.
4. Integra??o futura com uma interface de apoio a profissionais de sa?de.